In [2]:
import nba_api_module as nbpim
import pandas as pd

pd.set_option('display.max_columns', None)

TEST_GAME_ID = '0022400724'
TEST_TEAM_ID = 1610612760

In [ ]:
# Vegas PR calc basics

import pandas as pd
import nba_api_module as nbpim

games = nbpim.get_season_game_log(season='2024-25')

# Inicializáljuk a PR-eket minden csapatra
teams = games['TEAM_ID'].unique()
PR = {team_id: 0.0 for team_id in teams}

# Paraméterek
HCA = 3   # Home Court Advantage
K = 0.1   # learning rate

# Hozzáadunk két új oszlopot a várható marginhez és a PR frissítéshez
games['PR_home'] = 0.0
games['PR_away'] = 0.0
games['expected_margin'] = 0.0
games['beat_spread'] = 0.0

# jelöljük, melyik sor a hazai csapat
games['is_home'] = games['MATCHUP'].str.contains(' vs. ')

# home és away df-ek
home_df = games[games['is_home']].copy()
away_df = games[~games['is_home']].copy()

# merge Game ID alapján
merged = pd.merge(
    home_df,
    away_df,
    on='GAME_ID',
    suffixes=('_home', '_away')
)

# PR számítás
for idx, row in merged.iterrows():
    home_id = row['TEAM_ID_home']
    away_id = row['TEAM_ID_away']
    home_pts = row['PTS_home']
    away_pts = row['PTS_away']

    # várható margin
    expected_margin = PR[home_id] - PR[away_id] + HCA
    merged.at[idx, 'expected_margin'] = expected_margin

    # tényleges margin
    actual_margin = home_pts - away_pts
    beat_spread = actual_margin - expected_margin
    merged.at[idx, 'beat_spread'] = beat_spread

    # PR frissítés
    PR[home_id] += K * beat_spread
    PR[away_id] -= K * beat_spread

    merged.at[idx, 'PR_home'] = PR[home_id]
    merged.at[idx, 'PR_away'] = PR[away_id]

# merged df tartalmazza a számolt PR-eket, expected_margin-t és beat_spread-et
print(merged[['GAME_ID','TEAM_ABBREVIATION_home','TEAM_ABBREVIATION_away',
              'PR_home','PR_away','expected_margin','beat_spread']])


         GAME_ID TEAM_ABBREVIATION_home TEAM_ABBREVIATION_away    PR_home  \
0     0022400061                    BOS                    NYK   2.000000   
1     0022400062                    LAL                    MIN   0.400000   
2     0022400064                    ATL                    BKN   0.100000   
3     0022400069                    NOP                    CHI   0.900000   
4     0022400068                    HOU                    CHA  -0.800000   
...          ...                    ...                    ...        ...   
1220  0022401191                    PHI                    CHI -11.019905   
1221  0022401199                    POR                    LAL   2.258625   
1222  0022401200                    SAC                    PHX   1.412838   
1223  0022401197                    SAS                    TOR  -2.833981   
1224  0022401194                    MEM                    DAL   3.625474   

       PR_away  expected_margin  beat_spread  
0    -2.000000         3.000

In [ ]:
import nba_api_module as nbpim

games = nbpim.get_season_game_log(season='2024-25')

games['is_home'] = games['MATCHUP'].str.contains(' vs. ')

home_df = games[games['is_home']].copy()
away_df = games[~games['is_home']].copy()

merged = pd.merge(
    home_df,
    away_df,
    on='GAME_ID',
    suffixes=('_home', '_away')
)

merged = merged[merged['TEAM_ID_home'] != merged['TEAM_ID_away']]
merged

,SEASON_ID_home,TEAM_ID_home,TEAM_ABBREVIATION_home,TEAM_NAME_home,GAME_ID,GAME_DATE_home,MATCHUP_home,WL_home,MIN_home,FGM_home,FGA_home,FG_PCT_home,FG3M_home,FG3A_home,FG3_PCT_home,FTM_home,FTA_home,FT_PCT_home,OREB_home,DREB_home,REB_home,AST_home,STL_home,BLK_home,TOV_home,PF_home,PTS_home,PLUS_MINUS_home,VIDEO_AVAILABLE_home,is_home_home,SEASON_ID_away,TEAM_ID_away,TEAM_ABBREVIATION_away,TEAM_NAME_away,GAME_DATE_away,MATCHUP_away,WL_away,MIN_away,FGM_away,FGA_away,FG_PCT_away,FG3M_away,FG3A_away,FG3_PCT_away,FTM_away,FTA_away,FT_PCT_away,OREB_away,DREB_away,REB_away,AST_away,STL_away,BLK_away,TOV_away,PF_away,PTS_away,PLUS_MINUS_away,VIDEO_AVAILABLE_away,is_home_away
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,95,0.505,29,61,0.475,7,8,0.875,11,29,40,33,6,3,4,15,132,23,1,True,22024,1610612752,NYK,New York Knicks,2024-10-22,NYK @ BOS,L,240,43,78,0.551,11,30,0.367,12,16,0.750,5,29,34,20,2,3,12,12,109,-23,1,False
1,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,95,0.442,5,30,0.167,21,25,0.840,15,31,46,22,7,8,7,22,110,7,1,True,22024,1610612750,MIN,Minnesota Timberwolves,2024-10-22,MIN @ LAL,L,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,1,False
2,22024,1610612737,ATL,Atlanta Hawks,0022400064,2024-10-23,ATL vs. BKN,W,240,39,80,0.488,9,28,0.321,33,46,0.717,12,33,45,25,12,9,16,20,120,4,1,True,22024,1610612751,BKN,Brooklyn Nets,2024-10-23,BKN @ ATL,L,240,40,91,0.440,17,43,0.395,19,25,0.760,12,31,43,21,8,6,19,32,116,-4,1,False
3,22024,1610612740,NOP,New Orleans Pelicans,0022400069,2024-10-23,NOP vs. CHI,W,240,45,97,0.464,14,37,0.378,19,21,0.905,7,35,42,29,15,10,12,17,123,12,1,True,22024,1610612741,CHI,Chicago Bulls,2024-10-23,CHI @ NOP,L,240,42,86,0.488,10,34,0.294,17,25,0.680,9,38,47,26,4,2,21,16,111,-12,1,False
4,22024,1610612745,HOU,Houston Rockets,0022400068,2024-10-23,HOU vs. CHA,L,240,38,103,0.369,13,43,0.302,16,20,0.800,16,27,43,19,8,4,8,18,105,-5,1,True,22024,1610612766,CHA,Charlotte Hornets,2024-10-23,CHA @ HOU,W,240,38,85,0.447,15,39,0.385,19,24,0.792,15,41,56,20,6,10,17,21,110,5,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1220,22024,1610612755,PHI,Philadelphia 76ers,0022401191,2025-04-13,PHI vs. CHI,L,240,36,94,0.383,12,44,0.273,18,21,0.857,10,38,48,20,7,6,15,15,102,-20,1,True,22024,1610612741,CHI,Chicago Bulls,2025-04-13,CHI @ PHI,W,240,47,103,0.456,13,40,0.325,15,20,0.750,15,44,59,29,10,4,13,17,122,20,1,False
1221,22024,1610612757,POR,Portland Trail Blazers,0022401199,2025-04-13,POR vs. LAL,W,240,42,96,0.438,15,42,0.357,10,21,0.476,27,29,56,27,7,6,16,16,109,28,1,True,22024,1610612747,LAL,Los Angeles Lakers,2025-04-13,LAL @ POR,L,240,31,80,0.388,9,28,0.321,10,14,0.714,13,29,42,21,9,11,21,21,81,-28,1,False
1222,22024,1610612758,SAC,Sacramento Kings,0022401200,2025-04-13,SAC vs. PHX,W,240,46,88,0.523,14,34,0.412,3,3,1.000,10,33,43,31,6,4,13,7,109,11,1,True,22024,1610612756,PHX,Phoenix Suns,2025-04-13,PHX @ SAC,L,240,40,84,0.476,15,44,0.341,3,6,0.500,9,28,37,28,6,1,13,9,98,-11,1,False
1223,22024,1610612759,SAS,San Antonio Spurs,0022401197,2025-04-13,SAS vs. TOR,W,240,43,84,0.512,11,32,0.344,28,32,0.875,9,41,50,22,11,2,12,13,125,7,1,True,22024,1610612761,TOR,Toronto Raptors,2025-04-13,TOR @ SAS,L,240,46,97,0.474,14,39,0.359,12,16,0.750,12,30,42,32,8,1,13,23,118,-7,1,False


In [31]:
print(pd.merge(games, games, on='GAME_ID', suffixes=('_home', '_away')))

       GAME_ID  SEASON_ID_home  TEAM_ID_home TEAM_ABBREVIATION_home  \
0     22400001           22024    1610612737                    ATL   
1     22400002           22024    1610612748                    MIA   
2     22400003           22024    1610612753                    ORL   
3     22400004           22024    1610612752                    NYK   
4     22400005           22024    1610612749                    MIL   
...        ...             ...           ...                    ...   
1225  22401226           22024    1610612746                    LAC   
1226  22401227           22024    1610612752                    NYK   
1227  22401228           22024    1610612742                    DAL   
1228  22401229           22024    1610612737                    ATL   
1229  22401230           22024    1610612745                    HOU   

        TEAM_NAME_home GAME_DATE_home MATCHUP_home WL_home  MIN_home  \
0        Atlanta Hawks     2024-11-12    ATL @ BOS       W       240   
1  